In [4]:
# === SAFE LOAD CELL for Stage2FE-KCET ===
# Paste this at the top of your new Stage2FE-KCET.ipynb
import os, json, pickle, joblib
from pathlib import Path
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

OUT_DIR = Path(r"D:\Courses\Global Academy of Technology\kcet-college-pred\Version2\data\stage2_outputs")

print("Stage2 outputs folder:", OUT_DIR)
if not OUT_DIR.exists():
    raise FileNotFoundError(f"Stage2 outputs folder not found: {OUT_DIR}")

# Helpers
def safe_load_json(p: Path):
    if p.exists():
        try:
            return json.loads(p.read_text(encoding="utf8"))
        except Exception as e:
            print(f"Failed to load JSON {p}: {e}")
    return None

def safe_load_pickle(p: Path):
    if p.exists():
        try:
            with open(p, "rb") as f:
                return pickle.load(f)
        except Exception as e:
            print(f"Failed to load pickle {p}: {e}")
    return None

def safe_load_joblib(p: Path):
    if p.exists():
        try:
            return joblib.load(p)
        except Exception as e:
            print(f"Failed to load joblib {p}: {e}")
    return None

def safe_read_csv(p: Path):
    if p.exists():
        try:
            return pd.read_csv(p, low_memory=False)
        except Exception as e:
            print(f"Failed to read CSV {p}: {e}")
    return None

# Manifest (optional)
manifest = safe_load_json(OUT_DIR / "manifest.json")
if manifest:
    print("Loaded manifest.json (shows saved artifacts).")
else:
    print("No manifest.json found (continuing with best-effort load).")

# Expected artifact filenames (map to variable names)
expected = {
    # Dataframes / CSVs
    "stage2_dataset_csv": OUT_DIR / "stage2_dataset.csv",
    # standard pickles and joblibs
    "stage1_df": OUT_DIR / "stage1_df.pkl",
    "df_clean": OUT_DIR / "df_clean.pkl",
    "df_normalized": OUT_DIR / "df_normalized.pkl",
    "stage2_df_pkl": OUT_DIR / "stage2_df.pkl",
    "college_medians": OUT_DIR / "college_medians.pkl",
    "unstable_branches": OUT_DIR / "unstable_branches.pkl",
    # JSONs
    "year_max_ranks": OUT_DIR / "year_max_ranks.json",
    "tier_map": OUT_DIR / "tier_map.json",
    "tier_cutoffs": OUT_DIR / "tier_cutoffs.json",
    "feature_list": OUT_DIR / "feature_list.json",
    "college_tier_counts": OUT_DIR / "college_tier_counts.json",
    # encoders / scalers
    "scaler": OUT_DIR / "scaler.joblib",
    "rank_scaler": OUT_DIR / "rank_scaler.joblib",
    "label_encoders": OUT_DIR / "label_encoders.joblib",
    "target_encoders": OUT_DIR / "target_encoders.joblib",
}

# Container for loaded objects
_loaded = {}
_loaded_preview = {}

# 1) Load CSV stage2_dataset if present
stage2_df = safe_read_csv(expected["stage2_dataset_csv"])
if stage2_df is not None:
    _loaded["stage2_df"] = expected["stage2_dataset_csv"]
    _loaded_preview["stage2_df_head"] = stage2_df.head(5)

# 2) Load pickles
for name in ["stage1_df", "df_clean", "df_normalized", "stage2_df_pkl", "college_medians", "unstable_branches"]:
    p = expected[name]
    obj = safe_load_pickle(p)
    if obj is not None:
        varname = name
        # map stage2_df_pkl -> stage2_df for convenience
        if name == "stage2_df_pkl":
            varname = "stage2_df"
            stage2_df = obj
        else:
            globals()[name] = obj
        _loaded[varname] = str(p)
        if hasattr(obj, "head"):
            _loaded_preview[varname + "_head"] = obj.head(5)
        else:
            _loaded_preview[varname] = str(type(obj))

# 3) Load JSONs
for name in ["year_max_ranks", "tier_map", "tier_cutoffs", "feature_list", "college_tier_counts"]:
    p = expected.get(name)
    obj = safe_load_json(p)
    if obj is not None:
        globals()[name] = obj
        _loaded[name] = str(p)

# 4) Load joblib encoders/scalers
for name in ["scaler", "rank_scaler", "label_encoders", "target_encoders"]:
    p = expected.get(name)
    obj = safe_load_joblib(p)
    if obj is not None:
        globals()[name] = obj
        _loaded[name] = str(p)

# 5) Fallback: scan folder for common artifact patterns and attempt to load them
for p in OUT_DIR.glob("*"):
    if p.name in _loaded.values() or p.suffix in [".csv", ".pkl", ".joblib", ".json"]:
        # already loaded or will be handled above
        pass

# 6) Summary print
print("\n=== Loaded artifacts summary ===")
if _loaded:
    for k, v in _loaded.items():
        print(f"- {k:25s} : {v}")
else:
    print("No known artifacts loaded from expected list.")

print("\n=== Previews (if any) ===")
for k, v in _loaded_preview.items():
    print(f"\n-- {k} --")
    try:
        display(v)
    except Exception:
        print(v)

# 7) Hygiene: set stage2_df variable in namespace if only CSV was loaded
if 'stage2_df' not in globals() and 'stage2_df' in locals() and stage2_df is not None:
    globals()['stage2_df'] = stage2_df

# 8) Final helpful checks
print("\n=== Quick checks ===")
print("stage2_df loaded:", 'stage2_df' in globals() and globals()['stage2_df'] is not None)
print("college_medians loaded:", 'college_medians' in globals())
print("year_max_ranks loaded:", 'year_max_ranks' in globals())
print("tier_cutoffs loaded:", 'tier_cutoffs' in globals() or 'tier_map' in globals())
print("scaler loaded:", 'scaler' in globals())
print("\nIf some artifacts are missing, run Stage1 SAVE cell in the original notebook to produce them, then re-run this cell.")

# Expose loaded manifest for convenience
globals()['_stage2_outputs_manifest'] = manifest
print("\nLoaded manifest placed in variable _stage2_outputs_manifest (may be None).")


Stage2 outputs folder: D:\Courses\Global Academy of Technology\kcet-college-pred\Version2\data\stage2_outputs
Loaded manifest.json (shows saved artifacts).

=== Loaded artifacts summary ===
- stage1_df                 : D:\Courses\Global Academy of Technology\kcet-college-pred\Version2\data\stage2_outputs\stage1_df.pkl
- df_clean                  : D:\Courses\Global Academy of Technology\kcet-college-pred\Version2\data\stage2_outputs\df_clean.pkl
- college_medians           : D:\Courses\Global Academy of Technology\kcet-college-pred\Version2\data\stage2_outputs\college_medians.pkl
- year_max_ranks            : D:\Courses\Global Academy of Technology\kcet-college-pred\Version2\data\stage2_outputs\year_max_ranks.json
- tier_map                  : D:\Courses\Global Academy of Technology\kcet-college-pred\Version2\data\stage2_outputs\tier_map.json

=== Previews (if any) ===

-- stage1_df_head --


,College_Code,College_Name,Category,Branch,Cutoff_Rank,Year,Round,Exam_Type,Rank_Scaled,Branch_Norm,College_Clean,College_Final,Rank_norm,College_Tier
0,E001,University Visveswariah College of Engineering...,1G,CE Civil,29057.0,2020,1,KCET,0.189341,CE Civil,university visveswariah college of engineering...,university of visvesvaraya college of engineer...,0.189341,1
1,E001,University Visveswariah College of Engineering...,1K,CE Civil,30427.0,2020,1,KCET,0.198268,CE Civil,university visveswariah college of engineering...,university of visvesvaraya college of engineer...,0.198268,1
2,E001,University Visveswariah College of Engineering...,2AG,CE Civil,27430.0,2020,1,KCET,0.178739,CE Civil,university visveswariah college of engineering...,university of visvesvaraya college of engineer...,0.178739,1
3,E001,University Visveswariah College of Engineering...,2AR,CE Civil,28173.0,2020,1,KCET,0.183581,CE Civil,university visveswariah college of engineering...,university of visvesvaraya college of engineer...,0.183581,1
4,E001,University Visveswariah College of Engineering...,2BG,CE Civil,26255.0,2020,1,KCET,0.171082,CE Civil,university visveswariah college of engineering...,university of visvesvaraya college of engineer...,0.171082,1



-- df_clean_head --


,College_Code,College_Name,Category,Branch,Cutoff_Rank,Year,Round,Exam_Type,Rank_Scaled,Branch_Norm,College_Clean,College_Final
0,E001,University Visveswariah College of Engineering...,1G,CE Civil,29057.0,2020,1,KCET,0.189341,CE Civil,university visveswariah college of engineering...,university of visvesvaraya college of engineer...
1,E001,University Visveswariah College of Engineering...,1K,CE Civil,30427.0,2020,1,KCET,0.198268,CE Civil,university visveswariah college of engineering...,university of visvesvaraya college of engineer...
2,E001,University Visveswariah College of Engineering...,2AG,CE Civil,27430.0,2020,1,KCET,0.178739,CE Civil,university visveswariah college of engineering...,university of visvesvaraya college of engineer...
3,E001,University Visveswariah College of Engineering...,2AR,CE Civil,28173.0,2020,1,KCET,0.183581,CE Civil,university visveswariah college of engineering...,university of visvesvaraya college of engineer...
4,E001,University Visveswariah College of Engineering...,2BG,CE Civil,26255.0,2020,1,KCET,0.171082,CE Civil,university visveswariah college of engineering...,university of visvesvaraya college of engineer...



-- college_medians_head --


College_Code
E001    27110.5
E002    44555.0
E003    35799.0
E004    64236.0
E005     9543.0
Name: Cutoff_Rank, dtype: float64


=== Quick checks ===
stage2_df loaded: False
college_medians loaded: True
year_max_ranks loaded: True
tier_cutoffs loaded: True
scaler loaded: True

If some artifacts are missing, run Stage1 SAVE cell in the original notebook to produce them, then re-run this cell.

Loaded manifest placed in variable _stage2_outputs_manifest (may be None).


In [1]:
# === Stage2FE-KCET : CELL 1 ===
# Load Stage-1 artifacts, engineering-only dataset, and run no-leakage checks.
# Saves a Stage2 manifest & quick report in stage2_outputs/.
import pandas as pd
import json
from pathlib import Path
import joblib
import pickle
import warnings
warnings.filterwarnings("ignore")

# --- Paths (adjust if needed) ---
BASE = Path(r"D:\Courses\Global Academy of Technology\kcet-college-pred\Version2")
PROCESSED = BASE / "data" / "processed_data"
STAGE2_OUT = BASE / "data" / "stage2_outputs"
STAGE1_FILES = PROCESSED / "Stage1Files"
STAGE2_OUT.mkdir(parents=True, exist_ok=True)

# --- Files we expect (safe-load if present) ---
engineering_csv = PROCESSED / "KCET_split_engineering_only.csv"   # cleaned engineering dataset
stage2_csv = PROCESSED / "KCET_stage2_dataset.csv"               # earlier stage2 (if exists)
normalized_csv = STAGE1_FILES / "kcet_normalized.csv"            # normalized created earlier
manifest_path = STAGE2_OUT / "manifest.json"

# helper safe loaders
def safe_read_csv(p: Path):
    if p.exists():
        try:
            return pd.read_csv(p, low_memory=False)
        except Exception as e:
            raise RuntimeError(f"Failed to read CSV {p}: {e}")
    return None

def safe_load_json(p: Path):
    if p.exists():
        return json.loads(p.read_text(encoding="utf8"))
    return None

def safe_load_joblib(p: Path):
    if p.exists():
        return joblib.load(p)
    return None

def safe_load_pickle(p: Path):
    if p.exists():
        with open(p, "rb") as f:
            return pickle.load(f)
    return None

# --- Load primary dataframe (prefer engineering-only normalized file if present) ---
df = None
source_note = None

for candidate in [normalized_csv, PROCESSED / "KCET_stage2_dataset.csv", engineering_csv, PROCESSED / "KCET_split_cleaned_filtered.csv"]:
    df_candidate = safe_read_csv(candidate)
    if df_candidate is not None:
        df = df_candidate
        source_note = str(candidate)
        break

if df is None:
    raise FileNotFoundError(
        "No input dataset found. Expected one of:\n"
        f" - {normalized_csv}\n - {PROCESSED / 'KCET_stage2_dataset.csv'}\n - {engineering_csv}\n - {PROCESSED / 'KCET_split_cleaned_filtered.csv'}\n"
        "Place the engineering-only normalized CSV in processed_data/ or Stage1Files/ and re-run."
    )

print("Loaded dataset from:", source_note)
print("Rows:", len(df))
print("Columns:", list(df.columns))

# --- Basic sanity & dtype fixes (do not create leak features here) ---
expected_cols = ["College_Code","Branch","Category","Cutoff_Rank","Year","Round"]
missing = [c for c in expected_cols if c not in df.columns]
if missing:
    raise KeyError(f"Input dataset missing required columns: {missing}")

# Ensure types
df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype("Int64")
df["Round"] = pd.to_numeric(df["Round"], errors="coerce").astype("Int64")
df["Cutoff_Rank"] = pd.to_numeric(df["Cutoff_Rank"], errors="coerce")

# If normalized rank exists, check it's within [0,1]
if "Rank_norm" in df.columns:
    out_of_bounds = df.loc[(df["Rank_norm"] < 0) | (df["Rank_norm"] > 1)]
    print("Rank_norm present. values out of [0,1]:", len(out_of_bounds))
else:
    print("Rank_norm NOT present yet. Will be created explicitly in FE cell (per-year).")

# --- Load Stage-1 artifacts if present (non-blocking) ---
artifacts = {}
artifacts["year_max_ranks"] = safe_load_json(STAGE1_FILES / "year_max_ranks.json") or safe_load_json(STAGE2_OUT / "year_max_ranks.json")
artifacts["college_medians"] = safe_load_pickle(STAGE1_FILES / "college_medians.pkl") or safe_load_pickle(STAGE2_OUT / "college_medians.pkl")
artifacts["tier_map"] = safe_load_json(STAGE2_OUT / "tier_map.json") or safe_load_json(STAGE1_FILES / "tier_map.json") or safe_load_json(PROCESSED / "stage1kcet_cell1_report.csv".replace(".csv",".json"))
artifacts["unstable_branches"] = safe_load_pickle(STAGE2_OUT / "unstable_branches.pkl") or safe_load_pickle(STAGE1_FILES / "unstable_branches.pkl")

# --- Quick checks to prevent leakage in later FE ---
# 1) Ensure Year values are within expected range and consecutive
years = sorted(df["Year"].dropna().unique().tolist())
print("Years found:", years)
if len(years) < 2:
    raise RuntimeError("Insufficient number of years for temporal FE; at least 2 distinct years required.")

# 2) No future-year columns used: scan for suspicious columns (e.g., any column named '*_future*' or containing 'next_year')
suspect_cols = [c for c in df.columns if any(tok in c.lower() for tok in ["future","next_year","leak","target_", "y+1", "y_next"])]
if suspect_cols:
    raise RuntimeError(f"Found suspicious columns that may cause leakage: {suspect_cols}")

# 3) Check duplicates of key identifiers that could hide leakage
dups = df.duplicated(subset=["College_Code","Branch","Category","Year","Round","Cutoff_Rank"], keep=False).sum()
print("Exact key-duplicates (College+Branch+Category+Year+Round+Cutoff_Rank):", int(dups))

# 4) Basic distribution summary (no heavy computations)
summary = {
    "rows": int(len(df)),
    "years": years,
    "min_cutoff": float(df["Cutoff_Rank"].min()),
    "max_cutoff": float(df["Cutoff_Rank"].max()),
    "median_cutoff": float(df["Cutoff_Rank"].median()),
    "missing_cutoff": int(df["Cutoff_Rank"].isna().sum()),
    "unique_colleges": int(df["College_Code"].nunique()),
    "unique_branches": int(df["Branch"].nunique()),
    "unique_categories": int(df["Category"].nunique()),
}

print("\nQuick dataset summary:")
for k,v in summary.items():
    print(f" - {k}: {v}")

# --- Save a validation report (safe, human-readable) ---
report = {
    "source": source_note,
    "summary": summary,
    "artifacts_available": {k: (v is not None) for k,v in artifacts.items()},
    "leakage_checks": {
        "suspicious_cols": suspect_cols,
        "duplicate_key_count": int(dups)
    }
}
with open(STAGE2_OUT / "stage2_input_validation.json", "w", encoding="utf8") as f:
    json.dump(report, f, indent=2, default=str)

print("\nSaved input validation report →", STAGE2_OUT / "stage2_input_validation.json")

# --- Snapshot of top branches and counts (for quick human check) ---
branch_counts = df["Branch"].value_counts().reset_index().rename(columns={"index":"Branch","Branch":"count"})
branch_counts.head(50).to_csv(STAGE2_OUT / "branch_counts_preview.csv", index=False)
print("Saved branch_counts_preview.csv (top 50) to stage2_outputs.")

# Expose variables for downstream cells
globals()["stage2_df"] = df
globals()["_stage2_input_report"] = report
globals()["_stage2_branch_counts"] = branch_counts

print("\nCELL 1 complete. `stage2_df` is available in the notebook namespace.")
print("Reply with the printed summary or any errors; I will provide CELL 2 (no-leakage FE: year-wise normalization + tier assignment + basic historical aggregates).")


Loaded dataset from: D:\Courses\Global Academy of Technology\kcet-college-pred\Version2\data\processed_data\Stage1Files\kcet_normalized.csv
Rows: 212487
Columns: ['College_Code', 'College_Name', 'Category', 'Branch', 'Cutoff_Rank', 'Year', 'Round', 'Exam_Type', 'Rank_Scaled', 'Branch_Norm', 'College_Clean', 'College_Final', 'Rank_norm']
Rank_norm present. values out of [0,1]: 0
Years found: [2020, 2021, 2022, 2023, 2024]
Exact key-duplicates (College+Branch+Category+Year+Round+Cutoff_Rank): 0

Quick dataset summary:
 - rows: 212487
 - years: [2020, 2021, 2022, 2023, 2024]
 - min_cutoff: 93.0
 - max_cutoff: 274884.0
 - median_cutoff: 73303.0
 - missing_cutoff: 0
 - unique_colleges: 223
 - unique_branches: 150
 - unique_categories: 48

Saved input validation report → D:\Courses\Global Academy of Technology\kcet-college-pred\Version2\data\stage2_outputs\stage2_input_validation.json
Saved branch_counts_preview.csv (top 50) to stage2_outputs.

CELL 1 complete. `stage2_df` is available in th

In [3]:
# === Fix JSON serialization error and re-save stage2 globals (run this cell) ===
import json
from pathlib import Path
import numpy as np
import pandas as pd

BASE = Path(r"D:\Courses\Global Academy of Technology\kcet-college-pred\Version2")
STAGE2_OUT = BASE / "data" / "stage2_outputs"

# load the objects from variables in notebook if available, else try pickles
# we expect `tier_map`, `global_median`, `tier_meds` to be present in the notebook's namespace
# If not present, try to load from previously saved csv/pkl
try:
    tier_map  # noqa
except NameError:
    # try to load existing file (if any)
    p = STAGE2_OUT / "tier_map.json"
    if p.exists():
        with open(p, "r", encoding="utf8") as f:
            tier_map = json.load(f)
    else:
        tier_map = None

# global_median and tier_meds may be in memory as pandas types; if not, try to reconstruct
try:
    global_median  # noqa
except NameError:
    # try load from stage2_fe_step1.csv medians
    df_path = STAGE2_OUT / "stage2_fe_step1.csv"
    if df_path.exists():
        import pandas as pd
        tmp = pd.read_csv(df_path, low_memory=False)
        global_median = float(tmp["Cutoff_Rank"].median())
    else:
        global_median = None

try:
    tier_meds  # noqa
except NameError:
    # try to compute from stage2_df if present
    if 'stage2_df' in globals():
        tier_meds = stage2_df.groupby('College_Tier')['Cutoff_Rank'].median().to_dict()
    else:
        tier_meds = None

# Ensure tier_meds and tier_map have native python types for JSON
def to_native_keys(d):
    if d is None:
        return {}
    out = {}
    for k, v in d.items():
        # convert numpy/pandas ints to Python int, else str
        if isinstance(k, (np.integer, pd._libs.missing.NAType)):
            key = int(k)
        else:
            try:
                key = int(k)
            except Exception:
                key = str(k)
        # convert values to native python types (float/int/str)
        if isinstance(v, (np.floating, np.integer)):
            val = float(v) if isinstance(v, np.floating) else int(v)
        else:
            val = v
        out[key] = val
    return out

tier_meds_native = to_native_keys(tier_meds) if tier_meds is not None else {}
tier_map_native = to_native_keys(tier_map) if tier_map is not None else {}

# Save tier_map (if available)
if tier_map_native:
    with open(STAGE2_OUT / "tier_map.json", "w", encoding="utf8") as f:
        json.dump(tier_map_native, f, indent=2)
    print("Saved tier_map.json")

# Save stage2_globals.json with safe types
stage2_globals = {
    "global_median": float(global_median) if global_median is not None else None,
    "tier_meds": tier_meds_native
}
with open(STAGE2_OUT / "stage2_globals.json", "w", encoding="utf8") as f:
    json.dump(stage2_globals, f, indent=2)

print("Saved stage2_globals.json successfully at", STAGE2_OUT / "stage2_globals.json")


Saved tier_map.json
Saved stage2_globals.json successfully at D:\Courses\Global Academy of Technology\kcet-college-pred\Version2\data\stage2_outputs\stage2_globals.json


In [6]:
# === Stage2FE-KCET : CELL 2 (fixed - safe JSON keys) ===
# Create leak-free prior-year aggregates for college & branch and save artifacts safely.
import numpy as np
import pandas as pd
from pathlib import Path
import pickle, json

BASE = Path(r"D:\Courses\Global Academy of Technology\kcet-college-pred\Version2")
STAGE2_OUT = BASE / "data" / "stage2_outputs"
STAGE2_OUT.mkdir(parents=True, exist_ok=True)

# Use stage2_df prepared earlier by Cell 1
if 'stage2_df' not in globals():
    raise RuntimeError("stage2_df not found. Run Cell 1 first.")

df = stage2_df.copy()
print("Working rows:", len(df))

# Ensure Year numeric
df['Year'] = pd.to_numeric(df['Year'], errors='coerce').astype(int)

# -----------------------------
# Helper: compute prior summaries per group (college or branch)
# -----------------------------
def compute_group_year_stats(df, group_col, value_col='Cutoff_Rank'):
    gy = df.groupby([group_col, 'Year'])[value_col].agg(median='median', count='size').reset_index()
    gy = gy.sort_values([group_col, 'Year']).reset_index(drop=True)
    return gy

def compute_prior_medians_list(arr):
    res = [np.nan]
    for i in range(1, len(arr)):
        res.append(float(np.median(arr[:i])))
    return res

def compute_prior_counts_list(arr_counts):
    res = [0]
    for i in range(1, len(arr_counts)):
        res.append(int(sum(arr_counts[:i])))
    return res

def compute_prior_slope_list(years, medians, window=3):
    res = [np.nan]
    for i in range(1, len(medians)):
        prior_years = years[:i]
        prior_meds = medians[:i]
        if len(prior_meds) < 2:
            res.append(np.nan)
            continue
        ys = np.array(prior_years[-window:])
        ms = np.array(prior_meds[-window:])
        try:
            A = np.vstack([ys, np.ones_like(ys)]).T
            slope, intercept = np.linalg.lstsq(A, ms, rcond=None)[0]
            res.append(float(slope))
        except Exception:
            res.append(np.nan)
    return res

# -----------------------------
# College-level prior statistics
# -----------------------------
college_year = compute_group_year_stats(df, 'College_Code', 'Cutoff_Rank')
print("Unique college-years:", len(college_year))

college_prior_medians = []
college_prior_counts = []
college_prior_slope = []

for name, g in college_year.groupby('College_Code', sort=False):
    medians = g['median'].tolist()
    years = g['Year'].tolist()
    counts = g['count'].tolist()
    college_prior_medians.extend(compute_prior_medians_list(medians))
    college_prior_counts.extend(compute_prior_counts_list(counts))
    college_prior_slope.extend(compute_prior_slope_list(years, medians, window=3))

college_year['college_median_prev'] = college_prior_medians
college_year['college_pop_prev'] = college_prior_counts
college_year['college_trend_prev3_slope'] = college_prior_slope

# Merge back to main df (on College_Code & Year)
df = df.merge(
    college_year[['College_Code','Year','college_median_prev','college_pop_prev','college_trend_prev3_slope']],
    on=['College_Code','Year'], how='left'
)

# -----------------------------
# Branch-level prior statistics
# -----------------------------
branch_year = compute_group_year_stats(df, 'Branch', 'Cutoff_Rank')
branch_prior_medians = []
branch_prior_counts = []

for name, g in branch_year.groupby('Branch', sort=False):
    medians = g['median'].tolist()
    years = g['Year'].tolist()
    counts = g['count'].tolist()
    branch_prior_medians.extend(compute_prior_medians_list(medians))
    branch_prior_counts.extend(compute_prior_counts_list(counts))

branch_year['branch_median_prev'] = branch_prior_medians
branch_year['branch_pop_prev'] = branch_prior_counts

df = df.merge(
    branch_year[['Branch','Year','branch_median_prev','branch_pop_prev']],
    on=['Branch','Year'], how='left'
)

# -----------------------------
# Assign College Tier (use existing tier_map if available, else compute from college medians)
# -----------------------------
tier_map = None
if 'tier_map' in globals() and isinstance(tier_map, dict) and len(tier_map) > 0:
    tier_map = {k: int(v) for k, v in tier_map.items()}  # ensure native ints
    print("Using loaded tier_map from artifacts.")
else:
    college_meds_all = df.groupby('College_Code')['Cutoff_Rank'].median()
    p20 = float(np.percentile(college_meds_all, 20))
    p50 = float(np.percentile(college_meds_all, 50))
    print(f"Computed tier cutoffs from data: p20={int(p20)}, p50={int(p50)}")
    tier_map = {}
    for code, med in college_meds_all.items():
        if med <= p20:
            tier_map[code] = 1
        elif med <= p50:
            tier_map[code] = 2
        else:
            tier_map[code] = 3

# apply tier_map: create College_Tier column if missing
if 'College_Tier' not in df.columns:
    df['College_Tier'] = df['College_Code'].map(tier_map).astype('Int64')
else:
    df['College_Tier'] = df['College_Tier'].fillna(df['College_Code'].map(tier_map))

# -----------------------------
# Fallback imputation for prior medians / counts (no leakage)
# -----------------------------
global_median = float(df['Cutoff_Rank'].median())
tier_meds = df.groupby('College_Tier')['Cutoff_Rank'].median().to_dict()

def fill_prior_median(row, colname, tier_meds=tier_meds, global_median=global_median):
    val = row[colname]
    if not (pd.isna(val)):
        return val
    tier = row.get('College_Tier', None)
    try:
        tkey = int(tier) if pd.notna(tier) else None
    except Exception:
        tkey = None
    if tkey in tier_meds:
        return float(tier_meds[tkey])
    return global_median

df['college_median_prev_filled'] = df.apply(lambda r: fill_prior_median(r, 'college_median_prev'), axis=1)
df['branch_median_prev_filled'] = df.apply(lambda r: fill_prior_median(r, 'branch_median_prev'), axis=1)

df['college_pop_prev_filled'] = df['college_pop_prev'].fillna(0).astype(int)
df['branch_pop_prev_filled'] = df['branch_pop_prev'].fillna(0).astype(int)

# -----------------------------
# Save artifacts & outputs (convert keys/values to native python types for JSON)
# -----------------------------
college_year.to_csv(STAGE2_OUT / "college_year_medians_prior.csv", index=False)
branch_year.to_csv(STAGE2_OUT / "branch_year_medians_prior.csv", index=False)
df.to_csv(STAGE2_OUT / "stage2_fe_step1.csv", index=False)

def to_native(obj):
    """Recursively convert numpy/pandas types in dict keys/values to native python types for JSON."""
    if obj is None:
        return None
    # dict
    if isinstance(obj, dict):
        out = {}
        for k, v in obj.items():
            # convert key
            try:
                # try int
                if isinstance(k, (np.integer,)) or (isinstance(k, (str,)) and k.isdigit()):
                    newk = int(k)
                else:
                    newk = str(k)
            except Exception:
                newk = str(k)
            # convert value
            out[newk] = to_native(v)
        return out
    # list/tuple
    if isinstance(obj, (list, tuple)):
        return [to_native(v) for v in obj]
    # numpy scalar
    if isinstance(obj, (np.integer, np.int64, np.int32)):
        return int(obj)
    if isinstance(obj, (np.floating, np.float64, np.float32)):
        return float(obj)
    # pandas NA
    if pd.isna(obj):
        return None
    return obj

tier_map_native = to_native(tier_map)
tier_meds_native = to_native(tier_meds)

with open(STAGE2_OUT / "tier_map.json", "w", encoding="utf8") as f:
    json.dump(tier_map_native, f, indent=2)

with open(STAGE2_OUT / "stage2_globals.json", "w", encoding="utf8") as f:
    json.dump({"global_median": float(global_median), "tier_meds": tier_meds_native}, f, indent=2)

with open(STAGE2_OUT / "college_year_medians_prior.pkl", "wb") as f:
    pickle.dump(college_year, f)
with open(STAGE2_OUT / "branch_year_medians_prior.pkl", "wb") as f:
    pickle.dump(branch_year, f)

print("\nSaved artifacts to:", STAGE2_OUT)
print("stage2_fe_step1.csv rows:", sum(1 for _ in open(STAGE2_OUT / "stage2_fe_step1.csv"))-1)

# -----------------------------
# Quick diagnostics & summary
# -----------------------------
print("\nDiagnostic summary:")
print(" - Missing college_median_prev (after merge):", int(df['college_median_prev'].isna().sum()))
print(" - Missing branch_median_prev (after merge):", int(df['branch_median_prev'].isna().sum()))
print(" - college_pop_prev nulls:", int(df['college_pop_prev'].isna().sum()))
print(" - branch_pop_prev nulls:", int(df['branch_pop_prev'].isna().sum()))
print(" - college_median_prev_filled nulls:", int(df['college_median_prev_filled'].isna().sum()))
print(" - branch_median_prev_filled nulls:", int(df['branch_median_prev_filled'].isna().sum()))

print("\nTop rows of new features:")
display(df[['College_Code','Branch','Year','Cutoff_Rank','college_median_prev_filled','college_pop_prev_filled','branch_median_prev_filled','branch_pop_prev_filled','college_trend_prev3_slope']].head(12))

# expose for downstream
globals()['stage2_df'] = df
globals()['college_year_medians_prior'] = college_year
globals()['branch_year_medians_prior'] = branch_year
globals()['_stage2_fe_step1_path'] = STAGE2_OUT / "stage2_fe_step1.csv"

print("\nCELL 2 complete. Next: I will provide CELL 3 which will compute unstable-branch flags, log1p transforms, rank_norm validation, and small interaction features. Run Cell 2 and paste the diagnostics above, then I'll give Cell 3.")


Working rows: 212487
Unique college-years: 1039
Computed tier cutoffs from data: p20=56493, p50=87829

Saved artifacts to: D:\Courses\Global Academy of Technology\kcet-college-pred\Version2\data\stage2_outputs
stage2_fe_step1.csv rows: 212487

Diagnostic summary:
 - Missing college_median_prev (after merge): 27756
 - Missing branch_median_prev (after merge): 32490
 - college_pop_prev nulls: 0
 - branch_pop_prev nulls: 0
 - college_median_prev_filled nulls: 0
 - branch_median_prev_filled nulls: 0

Top rows of new features:


,College_Code,Branch,Year,Cutoff_Rank,college_median_prev_filled,college_pop_prev_filled,branch_median_prev_filled,branch_pop_prev_filled,college_trend_prev3_slope
0,E001,CE Civil,2020,29057.0,34522.0,0,34522.0,0,NaN
1,E001,CE Civil,2020,30427.0,34522.0,0,34522.0,0,NaN
2,E001,CE Civil,2020,27430.0,34522.0,0,34522.0,0,NaN
3,E001,CE Civil,2020,28173.0,34522.0,0,34522.0,0,NaN
4,E001,CE Civil,2020,26255.0,34522.0,0,34522.0,0,NaN
5,E001,CE Civil,2020,50152.0,34522.0,0,34522.0,0,NaN
6,E001,CE Civil,2020,22460.0,34522.0,0,34522.0,0,NaN
7,E001,CE Civil,2020,25836.0,34522.0,0,34522.0,0,NaN
8,E001,CE Civil,2020,24032.0,34522.0,0,34522.0,0,NaN
9,E001,CE Civil,2020,28109.0,34522.0,0,34522.0,0,NaN



CELL 2 complete. Next: I will provide CELL 3 which will compute unstable-branch flags, log1p transforms, rank_norm validation, and small interaction features. Run Cell 2 and paste the diagnostics above, then I'll give Cell 3.


In [8]:
# Fix and save diagnostics JSON (safe conversion of numpy/pandas types)
import json, numpy as np, pandas as pd
from pathlib import Path

BASE = Path(r"D:\Courses\Global Academy of Technology\kcet-college-pred\Version2")
STAGE2_OUT = BASE / "data" / "stage2_outputs"

# Use diag from namespace if present, else try to reconstruct minimal diag
try:
    diag  # noqa
except NameError:
    # fallback: build a minimal diag if possible
    import os
    diag = {
        "rows": int(len(globals().get('stage2_df', []))) if 'stage2_df' in globals() else None,
        "note": "diag not present originally; created fallback."
    }

def to_native(obj):
    # recursively convert numpy/pandas types to native python types
    if obj is None:
        return None
    if isinstance(obj, dict):
        out = {}
        for k, v in obj.items():
            # convert key to native type (int if possible)
            try:
                if isinstance(k, (np.integer,)) or (isinstance(k, (str,)) and k.isdigit()):
                    newk = int(k)
                else:
                    newk = str(k)
            except Exception:
                newk = str(k)
            out[newk] = to_native(v)
        return out
    if isinstance(obj, (list, tuple)):
        return [to_native(v) for v in obj]
    if isinstance(obj, (np.integer, np.int64, np.int32)):
        return int(obj)
    if isinstance(obj, (np.floating, np.float64, np.float32)):
        return float(obj)
    if pd.isna(obj):
        return None
    return obj

diag_safe = to_native(diag)

# Save
diag_path = STAGE2_OUT / "stage2_fe_diag.json"
with open(diag_path, "w", encoding="utf8") as f:
    json.dump(diag_safe, f, indent=2)

print("Saved diagnostics JSON at:", diag_path)

# Re-save unstable list safely if it exists
unstable_json = STAGE2_OUT / "unstable_branches_list.json"
if unstable_json.exists():
    try:
        data = json.loads(unstable_json.read_text(encoding="utf8"))
        with open(unstable_json, "w", encoding="utf8") as f:
            json.dump(to_native(data), f, indent=2)
        print("Re-saved unstable_branches_list.json safely.")
    except Exception:
        pass

# Re-save stage2_globals safely if exists
globals_path = STAGE2_OUT / "stage2_globals.json"
if globals_path.exists():
    try:
        g = json.loads(globals_path.read_text(encoding="utf8"))
        with open(globals_path, "w", encoding="utf8") as f:
            json.dump(to_native(g), f, indent=2)
        print("Re-saved stage2_globals.json safely.")
    except Exception:
        pass


Saved diagnostics JSON at: D:\Courses\Global Academy of Technology\kcet-college-pred\Version2\data\stage2_outputs\stage2_fe_diag.json
Re-saved unstable_branches_list.json safely.
Re-saved stage2_globals.json safely.


In [10]:
# === Stage2FE-KCET : CELL 3 (FINAL FIXED VERSION) ===
# - Validate / compute Rank_norm (per-year, no leakage)
# - Load or compute unstable branches, flag them
# - Apply log1p transforms
# - Add light interaction features
# - Save intermediate + final FE dataset
# - Save diagnostics with JSON-safe conversion

import pandas as pd
import numpy as np
from pathlib import Path
import pickle, json

BASE = Path(r"D:\Courses\Global Academy of Technology\kcet-college-pred\Version2")
PROCESSED = BASE / "data" / "processed_data"
OUT = BASE / "data" / "stage2_outputs"
OUT.mkdir(parents=True, exist_ok=True)

# ===============================
# Load Stage2 df (required)
# ===============================
if "stage2_df" in globals():
    df = stage2_df.copy()
else:
    p = OUT / "stage2_fe_step1.csv"
    if p.exists():
        df = pd.read_csv(p, low_memory=False)
    else:
        raise RuntimeError("ERROR: stage2_df not found and stage2_fe_step1.csv missing. Run Cell 2 first.")

print("Start rows:", len(df))

# ====================================================
# 1) RANK_NORM — VALIDATE / RECOMPUTE
# ====================================================
yr_path = OUT / "year_max_ranks.json"
if not yr_path.exists():
    alt = PROCESSED / "Stage1Files" / "year_max_ranks.json"
    if alt.exists():
        yr_path = alt

year_max = None
if yr_path.exists():
    with open(yr_path, "r", encoding="utf8") as f:
        loaded = json.load(f)
    # convert keys to int
    year_max = {int(k): float(v) for k, v in loaded.items()}
    print("Loaded year_max_ranks:", yr_path.name)

# Need to recompute Rank_norm?
needs_rank_norm = (
    ("Rank_norm" not in df.columns) or
    (df["Rank_norm"].isna().sum() > 0)
)

if needs_rank_norm:
    if year_max:
        df["Rank_norm"] = df.apply(
            lambda r: float(r["Cutoff_Rank"]) / year_max.get(int(r["Year"]), float(r["Cutoff_Rank"])),
            axis=1
        )
        print("Computed Rank_norm from year_max_ranks.")
    else:
        # fallback, safe per-year
        per_year_max = df.groupby("Year")["Cutoff_Rank"].transform("max")
        df["Rank_norm"] = df["Cutoff_Rank"] / per_year_max
        print("Computed Rank_norm from per-year max (fallback).")

# clamp 0..1
df["Rank_norm"] = df["Rank_norm"].clip(0, 1)

# ====================================================
# 2) LOAD / COMPUTE UNSTABLE BRANCH LIST
# ====================================================
unstable_pkl = OUT / "unstable_branches.pkl"
unstable_json = OUT / "unstable_branches_list.json"

unstable_list = None

# try json first
if unstable_json.exists():
    try:
        with open(unstable_json, "r", encoding="utf8") as f:
            unstable_list = json.load(f)
        print(f"Loaded unstable list from json ({len(unstable_list)} branches).")
    except:
        unstable_list = None

# try pkl
if unstable_list is None and unstable_pkl.exists():
    try:
        with open(unstable_pkl, "rb") as f:
            unstable_list = pickle.load(f)
        print(f"Loaded unstable list from pickle ({len(unstable_list)} branches).")
    except:
        unstable_list = None

# recompute conservative if still missing
if unstable_list is None:
    bstats = df.groupby("Branch").agg(
        total_count=("Branch","size"),
        unique_colleges=("College_Code","nunique"),
        median_cutoff=("Cutoff_Rank","median"),
        norm_std=("Rank_norm","std"),
        first_year=("Year","min")
    )
    cond = (
        (bstats["total_count"] < 150)
        | (bstats["unique_colleges"] <= 2)
        | (bstats["median_cutoff"] < 500)
        | (bstats["norm_std"].fillna(0) > 0.18)
        | (bstats["first_year"] >= 2022)
    )
    unstable_list = bstats[cond].index.tolist()

    # Save both formats
    with open(unstable_pkl, "wb") as f:
        pickle.dump(unstable_list, f)
    with open(unstable_json, "w", encoding="utf8") as f:
        json.dump(sorted(unstable_list), f, indent=2)

    print(f"Recomputed unstable branches (n={len(unstable_list)})")

# Add flag
df["branch_unstable_flag"] = df["Branch"].isin(unstable_list).astype(int)

# ====================================================
# 3) LOG TRANSFORMS
# ====================================================
df["Cutoff_log1p"] = np.log1p(df["Cutoff_Rank"].astype(float))
df["Rank_norm_log1p"] = np.log1p(df["Rank_norm"].astype(float) + 1e-9)

# ====================================================
# 4) SMALL INTERACTION FEATURES
# ====================================================
# ensure College_Tier exists
if "College_Tier" not in df.columns:
    # load tier_map
    tm = {}
    tmp = OUT / "tier_map.json"
    if tmp.exists():
        tm_raw = json.loads(tmp.read_text())
        # convert to str->int
        tm = {str(k): int(v) for k,v in tm_raw.items()}
    df["College_Tier"] = df["College_Code"].map(tm).astype("Int64").fillna(3)

df["rank_norm_x_tier"] = df["Rank_norm"] * df["College_Tier"].astype(float)

if "branch_pop_prev_filled" in df.columns:
    df["branchpop_x_tier"] = df["branch_pop_prev_filled"] * df["College_Tier"].astype(float)
else:
    df["branchpop_x_tier"] = 0.0

if "college_pop_prev_filled" in df.columns:
    df["collegepop_x_ranknorm"] = df["college_pop_prev_filled"] * df["Rank_norm"]
else:
    df["collegepop_x_ranknorm"] = 0.0

# ====================================================
# 5) DIAGNOSTICS (JSON-SAFE)
# ====================================================
diag = {
    "rows": int(len(df)),
    "unique_tiers": int(df["College_Tier"].nunique()),
    "tier_counts": {str(k): int(v) for k, v in df["College_Tier"].value_counts().items()},
    "unstable_branch_rows": int(df["branch_unstable_flag"].sum()),
    "cutoff_min": float(df["Cutoff_Rank"].min()),
    "cutoff_max": float(df["Cutoff_Rank"].max()),
    "ranknorm_min": float(df["Rank_norm"].min()),
    "ranknorm_max": float(df["Rank_norm"].max())
}

# save diag safely
with open(OUT / "stage2_fe_diag.json", "w", encoding="utf8") as f:
    json.dump(diag, f, indent=2)

print("Diagnostics saved:", diag)

# ====================================================
# 6) SAVE FINAL DATASETS
# ====================================================
intermediate = OUT / "stage2_fe_step2.csv"
final1 = PROCESSED / "KCET_stage2_final.csv"
final2 = OUT / "KCET_stage2_final.csv"

df.to_csv(intermediate, index=False)
df.to_csv(final1, index=False)
df.to_csv(final2, index=False)

# expose output
globals()["stage2_df"] = df
globals()["_stage2_final_csv"] = final1

print("Saved final Stage-2 FE →", final1)
print("Also saved in stage2_outputs/")

display(df.head(12))

print("\nCELL 3 COMPLETE. Ready for CELL 4 (encoding + scalers).")


Start rows: 212487
Loaded year_max_ranks: year_max_ranks.json
Loaded unstable list from json (149 branches).
Diagnostics saved: {'rows': 212487, 'unique_tiers': 3, 'tier_counts': {'1': 77261, '2': 76275, '3': 58951}, 'unstable_branch_rows': 212076, 'cutoff_min': 93.0, 'cutoff_max': 274884.0, 'ranknorm_min': 0.0005076142131979, 'ranknorm_max': 1.0}
Saved final Stage-2 FE → D:\Courses\Global Academy of Technology\kcet-college-pred\Version2\data\processed_data\KCET_stage2_final.csv
Also saved in stage2_outputs/


,College_Code,College_Name,Category,Branch,Cutoff_Rank,Year,Round,Exam_Type,Rank_Scaled,Branch_Norm,...,college_median_prev_filled,branch_median_prev_filled,college_pop_prev_filled,branch_pop_prev_filled,branch_unstable_flag,Cutoff_log1p,Rank_norm_log1p,rank_norm_x_tier,branchpop_x_tier,collegepop_x_ranknorm
0,E001,University Visveswariah College of Engineering...,1G,CE Civil,29057.0,2020,1,KCET,0.189341,CE Civil,...,34522.0,34522.0,0,0,1,10.277049,0.173399,0.189341,0.0,0.0
1,E001,University Visveswariah College of Engineering...,1K,CE Civil,30427.0,2020,1,KCET,0.198268,CE Civil,...,34522.0,34522.0,0,0,1,10.323119,0.180877,0.198268,0.0,0.0
2,E001,University Visveswariah College of Engineering...,2AG,CE Civil,27430.0,2020,1,KCET,0.178739,CE Civil,...,34522.0,34522.0,0,0,1,10.219429,0.164445,0.178739,0.0,0.0
3,E001,University Visveswariah College of Engineering...,2AR,CE Civil,28173.0,2020,1,KCET,0.183581,CE Civil,...,34522.0,34522.0,0,0,1,10.246155,0.168544,0.183581,0.0,0.0
4,E001,University Visveswariah College of Engineering...,2BG,CE Civil,26255.0,2020,1,KCET,0.171082,CE Civil,...,34522.0,34522.0,0,0,1,10.175650,0.157929,0.171082,0.0,0.0
5,E001,University Visveswariah College of Engineering...,2BR,CE Civil,50152.0,2020,1,KCET,0.326800,CE Civil,...,34522.0,34522.0,0,0,1,10.822834,0.282770,0.326800,0.0,0.0
6,E001,University Visveswariah College of Engineering...,3AG,CE Civil,22460.0,2020,1,KCET,0.146354,CE Civil,...,34522.0,34522.0,0,0,1,10.019536,0.136586,0.146354,0.0,0.0
7,E001,University Visveswariah College of Engineering...,3AR,CE Civil,25836.0,2020,1,KCET,0.168352,CE Civil,...,34522.0,34522.0,0,0,1,10.159563,0.155594,0.168352,0.0,0.0
8,E001,University Visveswariah College of Engineering...,3BG,CE Civil,24032.0,2020,1,KCET,0.156597,CE Civil,...,34522.0,34522.0,0,0,1,10.087183,0.145482,0.156597,0.0,0.0
9,E001,University Visveswariah College of Engineering...,3BK,CE Civil,28109.0,2020,1,KCET,0.183163,CE Civil,...,34522.0,34522.0,0,0,1,10.243881,0.168192,0.183163,0.0,0.0



CELL 3 COMPLETE. Ready for CELL 4 (encoding + scalers).


In [11]:
# === Stage2FE-KCET : CELL 4 ===
# Temporal target-encoding for high-cardinality features (college, branch, category)
# + smoothing, imputation, and saving encoders
# + Fit a StandardScaler on numeric features using years < max_year (no leakage)
# + Save artifacts to stage2_outputs
import pandas as pd
import numpy as np
from pathlib import Path
import pickle, json
from sklearn.preprocessing import StandardScaler
import joblib

BASE = Path(r"D:\Courses\Global Academy of Technology\kcet-college-pred\Version2")
PROCESSED = BASE / "data" / "processed_data"
OUT = BASE / "data" / "stage2_outputs"
OUT.mkdir(parents=True, exist_ok=True)

# Load df (result of Cell 3)
if "stage2_df" in globals():
    df = stage2_df.copy()
else:
    p = OUT / "KCET_stage2_final.csv"
    if p.exists():
        df = pd.read_csv(p, low_memory=False)
    else:
        raise RuntimeError("stage2_df not found. Run prior cells first.")

print("Rows for encoding:", len(df))

# Determine max_year (we'll treat max_year as holdout; scaler trained on years < max_year)
years = sorted(df['Year'].dropna().unique().astype(int).tolist())
max_year = years[-1]
train_years_mask = df['Year'] < max_year
print("Years found:", years, " -> max_year (holdout) =", max_year)
print("Rows used to fit scalers / aggregate stats (year < max_year):", int(train_years_mask.sum()))

# Global target for smoothing
global_mean = float(df.loc[train_years_mask, "Cutoff_Rank"].mean()) if train_years_mask.any() else float(df["Cutoff_Rank"].mean())
print("Global mean (used as prior):", int(global_mean))

# Smoothing parameter
SMOOTH = 10.0  # higher => stronger shrinkage toward global_mean; tune later in Stage-3

# Helper: compute temporal smoothed mean encoding for a feature
def temporal_smoothed_encoding(df, feature, target="Cutoff_Rank", smooth=SMOOTH):
    """
    Returns:
      enc_map: dict mapping year -> { feature_value : smoothed_mean_using rows with year < year }
      enc_col: a Series of encoded values aligned to df (using prior-year encodings)
    """
    enc_map = {}
    enc_col = pd.Series(index=df.index, dtype=float)

    # compute group-year sums and counts using only training years? We'll use all years but compute prior-only per year below
    # Get unique years sorted
    yrs = sorted(df['Year'].dropna().unique().astype(int).tolist())
    for y in yrs:
        # training data for this target year = rows with Year < y
        prior = df[df['Year'] < y]
        if prior.shape[0] == 0:
            enc_map[y] = {}  # no prior info
        else:
            grp = prior.groupby(feature)[target].agg(['sum','count']).reset_index()
            # smoothed mean = (sum + m * global_mean) / (count + m)
            grp['smoothed'] = (grp['sum'] + smooth * global_mean) / (grp['count'] + smooth)
            # convert to dict
            enc_map[y] = dict(zip(grp[feature].astype(str), grp['smoothed'].astype(float)))

        # for rows in year==y, fill enc_col using enc_map[y]
        mask = df['Year'] == y
        if mask.any():
            # map with fallback NaN (we will impute later)
            enc_col.loc[mask] = df.loc[mask, feature].astype(str).map(enc_map[y]).astype(float)

    return enc_map, enc_col

# Features to encode
high_card_feats = ["College_Code", "Branch", "Category"]

encoders = {}
for feat in high_card_feats:
    print(f"\nEncoding feature: {feat}")
    enc_map, enc_col = temporal_smoothed_encoding(df, feat, target="Cutoff_Rank", smooth=SMOOTH)
    encoders[feat] = enc_map
    df[f"{feat}_enc_prev"] = enc_col.values

    # Save per-feature per-year mapping as JSON-friendly (keys -> str)
    safe_map = {}
    for y, mp in enc_map.items():
        safe_map[int(y)] = {str(k): float(v) for k, v in mp.items()}
    with open(OUT / f"enc_{feat}_temporal.json", "w", encoding="utf8") as f:
        json.dump(safe_map, f, indent=2)

    # diagnostics
    missing_count = int(df[f"{feat}_enc_prev"].isna().sum())
    print(f"  Encoded; NaNs (no prior info for this value in prior years): {missing_count}")

# Imputation for encoded features: use tier-based median encoding or global_mean
# We'll use college_tier: compute enc median by tier from training years encodings (safe)
if 'College_Tier' in df.columns:
    tier_enc_medians = {}
    for feat in high_card_feats:
        tmp = df.loc[train_years_mask, :].copy()
        # for rows where encoding exists, map Tier -> median(enc)
        tmp = tmp.loc[tmp[f"{feat}_enc_prev"].notna()]
        tier_med = tmp.groupby('College_Tier')[f"{feat}_enc_prev"].median().to_dict()
        # convert keys to native ints
        tier_enc_medians[feat] = {int(k): float(v) for k, v in tier_med.items()}
else:
    tier_enc_medians = {feat: {} for feat in high_card_feats}

# Fill missing encodings with tier median if available else global_mean
for feat in high_card_feats:
    col = f"{feat}_enc_prev"
    before_na = int(df[col].isna().sum())
    def impute_val(row):
        val = row[col]
        if not pd.isna(val):
            return val
        tier = row.get('College_Tier', None)
        try:
            tier_k = int(tier) if pd.notna(tier) else None
        except Exception:
            tier_k = None
        if tier_k in tier_enc_medians.get(feat, {}):
            return float(tier_enc_medians[feat][tier_k])
        return float(global_mean)
    df[col + "_filled"] = df.apply(impute_val, axis=1)
    after_na = int(df[f"{col}_filled"].isna().sum())
    print(f"Filled {col}: before_na={before_na} after_na={after_na}")

# Save encoder artifacts (pickles + json)
with open(OUT / "temporal_encoders.pkl", "wb") as f:
    pickle.dump(encoders, f)

with open(OUT / "tier_enc_medians.json", "w", encoding="utf8") as f:
    json.dump({k: {str(kk): float(vv) for kk,vv in vv.items()} for k,v in tier_enc_medians.items() for kk,vv in {k:v}.items()}, f, indent=2)

print("\nSaved temporal encoder JSONs and pickle to stage2_outputs/")

# ----------------------------
# Fit StandardScaler on numeric features (no leakage): use years < max_year
# Numeric features to scale: Rank_norm, Cutoff_log1p, Rank_norm_log1p, college/branch encodings (filled)
# ----------------------------
num_feats = [
    "Rank_norm", "Cutoff_log1p", "Rank_norm_log1p",
    "College_Code_enc_prev_filled", "Branch_enc_prev_filled", "Category_enc_prev_filled",
    "college_pop_prev_filled", "branch_pop_prev_filled", "college_trend_prev3_slope"
]
# Ensure columns exist, create if missing
for c in num_feats:
    if c not in df.columns:
        df[c] = 0.0

scaler = StandardScaler()
scaler_fit_mask = df['Year'] < max_year
if scaler_fit_mask.sum() == 0:
    # fallback to all data
    scaler_fit_mask = df.index.to_series().astype(bool)

scaler.fit(df.loc[scaler_fit_mask, num_feats].fillna(0.0))
# Save scaler
joblib.dump(scaler, OUT / "stage2_numeric_scaler.joblib")
print("Fitted StandardScaler on numeric features using years <", max_year, "and saved to stage2_outputs.")

# Create scaled columns (but keep originals too)
scaled = scaler.transform(df[num_feats].fillna(0.0))
scaled_df = pd.DataFrame(scaled, columns=[c + "_scaled" for c in num_feats], index=df.index)
df = pd.concat([df, scaled_df], axis=1)

# ----------------------------
# Final save
# ----------------------------
final_path = PROCESSED / "KCET_stage2_final_with_encoders.csv"
out_path = OUT / "KCET_stage2_final_with_encoders.csv"
df.to_csv(final_path, index=False)
df.to_csv(out_path, index=False)

# Save manifest of saved files
saved = {
    "temporal_encoders_pickle": str(OUT / "temporal_encoders.pkl"),
    "enc_College_Code_json": str(OUT / "enc_College_Code_temporal.json") if (OUT / "enc_College_Code_temporal.json").exists() else "enc_College_Code_temporal.json not found",
    "enc_Branch_json": str(OUT / "enc_Branch_temporal.json") if (OUT / "enc_Branch_temporal.json").exists() else "enc_Branch_temporal.json not found",
    "enc_Category_json": str(OUT / "enc_Category_temporal.json") if (OUT / "enc_Category_temporal.json").exists() else "enc_Category_temporal.json not found",
    "tier_enc_medians": str(OUT / "tier_enc_medians.json"),
    "scaler_joblib": str(OUT / "stage2_numeric_scaler.joblib"),
    "final_csv_processed": str(final_path),
    "final_csv_stage2out": str(out_path)
}
# write manifest JSON safely
with open(OUT / "stage2_encoders_manifest.json", "w", encoding="utf8") as f:
    json.dump(saved, f, indent=2)

print("\nSaved final FE with encodings and scaler to:")
print(" - processed_data:", final_path)
print(" - stage2_outputs:", out_path)
print("\nCELL 4 complete. Exposed variables: 'stage2_df' updated with encodings and scaled cols, 'scaler' saved to stage2_outputs.")


Rows for encoding: 212487
Years found: [2020, 2021, 2022, 2023, 2024]  -> max_year (holdout) = 2024
Rows used to fit scalers / aggregate stats (year < max_year): 155511
Global mean (used as prior): 74044

Encoding feature: College_Code
  Encoded; NaNs (no prior info for this value in prior years): 27756

Encoding feature: Branch
  Encoded; NaNs (no prior info for this value in prior years): 32490

Encoding feature: Category
  Encoded; NaNs (no prior info for this value in prior years): 25191
Filled College_Code_enc_prev: before_na=27756 after_na=0
Filled Branch_enc_prev: before_na=32490 after_na=0
Filled Category_enc_prev: before_na=25191 after_na=0

Saved temporal encoder JSONs and pickle to stage2_outputs/
Fitted StandardScaler on numeric features using years < 2024 and saved to stage2_outputs.

Saved final FE with encodings and scaler to:
 - processed_data: D:\Courses\Global Academy of Technology\kcet-college-pred\Version2\data\processed_data\KCET_stage2_final_with_encoders.csv
 - s

In [1]:
# === Stage2FE-KCET : CELL 5 ===
# - Create Tier-specific CSVs
# - Create temporal train/val/test splits per Tier
# - Save final feature list & manifest (JSON-safe)
import pandas as pd
import numpy as np
import json
import pickle
from pathlib import Path
import math

BASE = Path(r"D:\Courses\Global Academy of Technology\kcet-college-pred\Version2")
PROCESSED = BASE / "data" / "processed_data"
OUT = BASE / "data" / "stage2_outputs"
PROCESSED.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

# Load prepared DF
if 'stage2_df' in globals():
    df = stage2_df.copy()
else:
    p = OUT / "KCET_stage2_final_with_encoders.csv"
    if p.exists():
        df = pd.read_csv(p, low_memory=False)
    else:
        raise RuntimeError("stage2_df not found in namespace and final file not present. Run prior cells first.")

print("Rows available for final split:", len(df))

# Ensure College_Tier and Year exist and are correct types
if 'College_Tier' not in df.columns:
    df['College_Tier'] = df.get('College_Tier', 3).astype('Int64')
else:
    df['College_Tier'] = pd.to_numeric(df['College_Tier'], errors='coerce').fillna(3).astype('Int64')

df['Year'] = pd.to_numeric(df['Year'], errors='coerce').astype(int)

# Final feature list (engineer-curated; these are safe — all created from <=Y only)
# Keep this list conservative. You can extend in Stage-3.
feature_list = [
    # rank / target transforms
    "Cutoff_Rank", "Cutoff_log1p", "Rank_norm", "Rank_norm_log1p",
    # encodings (filled)
    "College_Code_enc_prev_filled", "Branch_enc_prev_filled", "Category_enc_prev_filled",
    # prior popularity & trend
    "college_pop_prev_filled", "branch_pop_prev_filled", "college_trend_prev3_slope",
    # flags / interactions
    "branch_unstable_flag", "rank_norm_x_tier", "branchpop_x_tier", "collegepop_x_ranknorm",
    # context
    "Year", "Round", "College_Tier"
]

# Add scaled numeric columns if present
scaled_cols = [c for c in df.columns if c.endswith("_scaled")]
# keep scaled columns that correspond to numeric features used in modeling
for s in scaled_cols:
    if s not in feature_list:
        feature_list.append(s)

# Save feature_list
feature_list_path = OUT / "stage2_feature_list.json"
with open(feature_list_path, "w", encoding="utf8") as f:
    json.dump({"features": feature_list}, f, indent=2)

# ---------- Tier datasets ----------
tiered_info = {}
for tier in sorted(df['College_Tier'].dropna().unique().astype(int).tolist()):
    tier_df = df[df['College_Tier'] == int(tier)].copy()
    tier_name = f"Tier{int(tier)}"
    # save CSV
    path_proc = PROCESSED / f"KCET_stage2_{tier_name}.csv"
    path_out = OUT / f"KCET_stage2_{tier_name}.csv"
    tier_df.to_csv(path_proc, index=False)
    tier_df.to_csv(path_out, index=False)
    tiered_info[tier_name] = {
        "rows": int(len(tier_df)),
        "processed_path": str(path_proc),
        "stage2_out_path": str(path_out)
    }

# ---------- Temporal splits ----------
# Define years
years = sorted(df['Year'].dropna().unique().astype(int).tolist())
if len(years) < 3:
    raise RuntimeError("Not enough distinct years for recommended temporal split (need >=3).")

train_years = [y for y in years if y <= (years[-1] - 2)]  # e.g., 2020-2022 when last=2024
val_year = years[-2]  # e.g., 2023
test_year = years[-1]  # e.g., 2024

print("Temporal split => train_years:", train_years, "val:", val_year, "test:", test_year)

split_info = {}

for tier in sorted(df['College_Tier'].dropna().unique().astype(int).tolist()):
    tier_df = df[df['College_Tier'] == int(tier)].copy()
    train_df = tier_df[tier_df['Year'].isin(train_years)].copy()
    val_df = tier_df[tier_df['Year'] == val_year].copy()
    test_df = tier_df[tier_df['Year'] == test_year].copy()

    # Save
    tdir = OUT / f"Tier{tier}"
    tdir.mkdir(parents=True, exist_ok=True)

    train_p = tdir / f"tier{tier}_train.csv"
    val_p = tdir / f"tier{tier}_val.csv"
    test_p = tdir / f"tier{tier}_test.csv"

    train_df.to_csv(train_p, index=False)
    val_df.to_csv(val_p, index=False)
    test_df.to_csv(test_p, index=False)

    split_info[f"Tier{tier}"] = {
        "train_rows": int(len(train_df)),
        "val_rows": int(len(val_df)),
        "test_rows": int(len(test_df)),
        "train_path": str(train_p),
        "val_path": str(val_p),
        "test_path": str(test_p)
    }

# ---------- Final manifest & counts ----------
manifest = {
    "total_rows": int(len(df)),
    "years": years,
    "train_years": train_years,
    "val_year": int(val_year),
    "test_year": int(test_year),
    "feature_list_path": str(feature_list_path),
    "tier_files": tiered_info,
    "tier_splits": split_info
}

# JSON-safe convert (ensure ints / strs)
def to_native(o):
    if o is None:
        return None
    if isinstance(o, dict):
        return { (int(k) if isinstance(k,(np.integer,)) else str(k)) : to_native(v) for k,v in o.items()}
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, list):
        return [to_native(x) for x in o]
    return o

manifest_safe = to_native(manifest)

with open(OUT / "stage2_final_manifest.json", "w", encoding="utf8") as f:
    json.dump(manifest_safe, f, indent=2)

# Also save a compact CSV with selected features for each tier (useful for Stage-3 quick loading)
for tier in sorted(df['College_Tier'].dropna().unique().astype(int).tolist()):
    tier_df = df[df['College_Tier'] == int(tier)].copy()
    sel = [c for c in feature_list if c in tier_df.columns]
    tier_df[sel + ["College_Code","Branch","Category","College_Name"]].to_csv(PROCESSED / f"KCET_stage2_tier{tier}_features.csv", index=False)
    tier_df[sel + ["College_Code","Branch","Category","College_Name"]].to_csv(OUT / f"KCET_stage2_tier{tier}_features.csv", index=False)

print("\nSaved tier CSVs and temporal splits.")
print("Feature list saved:", feature_list_path)
print("Manifest saved:", OUT / "stage2_final_manifest.json")

# Print summary
print("\nSummary per tier (rows):")
for k,v in tiered_info.items():
    print(f" - {k}: {v['rows']} rows")

print("\nTemporal split sizes (train/val/test) per tier:")
for k,v in split_info.items():
    print(f" - {k}: train {v['train_rows']}, val {v['val_rows']}, test {v['test_rows']}")

# expose manifest and feature list for downstream
globals()['_stage2_feature_list'] = feature_list
globals()['_stage2_final_manifest'] = manifest_safe

print("\nCELL 5 complete. Next: Stage-3 notebook/template or model training cells when you're ready.")


Rows available for final split: 212487
Temporal split => train_years: [2020, 2021, 2022] val: 2023 test: 2024

Saved tier CSVs and temporal splits.
Feature list saved: D:\Courses\Global Academy of Technology\kcet-college-pred\Version2\data\stage2_outputs\stage2_feature_list.json
Manifest saved: D:\Courses\Global Academy of Technology\kcet-college-pred\Version2\data\stage2_outputs\stage2_final_manifest.json

Summary per tier (rows):
 - Tier1: 77261 rows
 - Tier2: 76275 rows
 - Tier3: 58951 rows

Temporal split sizes (train/val/test) per tier:
 - Tier1: train 42541, val 16352, test 18368
 - Tier2: train 38399, val 18280, test 19596
 - Tier3: train 24872, val 15067, test 19012

CELL 5 complete. Next: Stage-3 notebook/template or model training cells when you're ready.
